# OTUS DZ07 — Multi-Agent Travel Assistant
Минимальный LangGraph-прототип: Manager делегирует поиск правил Searcher/PolicyRAG и возвращает результат. Внешние API не нужны.

In [ ]:
%pip -q install 'langgraph>=0.2,<2'

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

POLICY = {
    'hotel': 'Москва/СПб: до 12 000 ₽ за ночь [TRAVEL-POLICY-001#HOTEL]',
    'flight': 'По России базовый класс — эконом [TRAVEL-POLICY-001#FLIGHT]',
}

class State(TypedDict, total=False):
    request: str
    messages: list[str]
    evidence: list[str]
    answer: str

def manager(state):
    return {'messages': state.get('messages', []) + ['Manager → Searcher: найди правила для поездки в Москву']}

def searcher(state):
    hits = [POLICY['hotel'], POLICY['flight']]
    return {'evidence': hits, 'messages': state['messages'] + ['Searcher → Manager: правила найдены']}

def finalize(state):
    text = 'Можно планировать эконом-перелёт и гостиницу до 12 000 ₽/ночь. Источники: ' + '; '.join(state['evidence'])
    return {'answer': text, 'messages': state['messages'] + ['Manager: сформирован итоговый ответ']}

g = StateGraph(State)
g.add_node('manager', manager)
g.add_node('searcher', searcher)
g.add_node('finalize', finalize)
g.add_edge(START, 'manager')
g.add_edge('manager', 'searcher')
g.add_edge('searcher', 'finalize')
g.add_edge('finalize', END)
app = g.compile()
result = app.invoke({'request': 'Командировка Санкт-Петербург → Москва', 'messages': []})
print('\n'.join(result['messages']))
print('\nANSWER:', result['answer'])